In [1]:
"""
DATATHON 2026 — Round 1
Part 1: Multiple Choice Questions — Solution Code
"""

import pandas as pd
import numpy as np

# ── Load all required files ──────────────────────────────────────────────────
orders       = pd.read_csv('orders.csv',      parse_dates=['order_date'])
order_items  = pd.read_csv('order_items.csv', low_memory=False)
products     = pd.read_csv('products.csv')
customers    = pd.read_csv('customers.csv')
returns      = pd.read_csv('returns.csv')
web_traffic  = pd.read_csv('web_traffic.csv')
payments     = pd.read_csv('payments.csv')
geography    = pd.read_csv('geography.csv')
sales        = pd.read_csv('sales.csv',       parse_dates=['Date'])
print("Done.\n")

Done.



In [2]:
# ── Q1 ───────────────────────────────────────────────────────────────────────
print("Q1: Median inter-order gap for multi-order customers")
multi_order = orders.groupby('customer_id').filter(lambda x: len(x) > 1)
gaps = (
    multi_order
    .sort_values(['customer_id', 'order_date'])
    .groupby('customer_id')['order_date']
    .apply(lambda x: x.diff().dt.days.dropna())
    .reset_index(drop=True)
)
q1_answer = gaps.median()
print(f"  Median gap: {q1_answer:.1f} days")

Q1: Median inter-order gap for multi-order customers
  Median gap: 144.0 days


In [3]:
# ── Q2 ───────────────────────────────────────────────────────────────────────
print("Q2: Segment with highest average gross margin")
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
gm_by_seg = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print(gm_by_seg.to_string())

Q2: Segment with highest average gross margin
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343


In [4]:
# ── Q3 ───────────────────────────────────────────────────────────────────────
print("Q3: Most common return reason in Streetwear category")
streetwear_ids = products.loc[products['category'] == 'Streetwear', 'product_id']
streetwear_returns = returns[returns['product_id'].isin(streetwear_ids)]
top_reason = streetwear_returns['return_reason'].value_counts()
print(top_reason.to_string())

Q3: Most common return reason in Streetwear category
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159


In [5]:
# ── Q4 ───────────────────────────────────────────────────────────────────────
print("Q4: Traffic source with lowest avg bounce rate")
br_by_source = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print(br_by_source.to_string())

Q4: Traffic source with lowest avg bounce rate
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511


In [6]:
# ── Q5 ───────────────────────────────────────────────────────────────────────
print("Q5: percent of order_items with promo_id not null")
pct_promo = order_items['promo_id'].notna().mean() * 100
print(f"  Percentage with promo: {pct_promo:.1f}%")

Q5: percent of order_items with promo_id not null
  Percentage with promo: 38.7%


In [7]:
# ── Q6 ───────────────────────────────────────────────────────────────────────
print("Q6: Age group with highest avg orders per customer")
cust_nn = customers[customers['age_group'].notna()]
orders_per_cust = orders.groupby('customer_id').size().reset_index(name='order_count')
merged = cust_nn.merge(orders_per_cust, on='customer_id', how='left')
merged['order_count'] = merged['order_count'].fillna(0)
avg_by_age = merged.groupby('age_group')['order_count'].mean().sort_values(ascending=False)
print(avg_by_age.to_string())

Q6: Age group with highest avg orders per customer
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656


In [8]:
# ── Q7 ───────────────────────────────────────────────────────────────────────
print("Q7: Region with highest total revenue")
order_revenue = (
    order_items
    .assign(line_revenue=lambda df: df['quantity'] * df['unit_price'] - df['discount_amount'])
    .groupby('order_id')['line_revenue']
    .sum()
    .reset_index(name='revenue')
    )
orders_geo = orders.merge(geography[['zip', 'region']], on='zip', how='left')
orders_geo_rev = orders_geo.merge(order_revenue, on='order_id', how='left')
rev_by_region = orders_geo_rev.groupby('region')['revenue'].sum().sort_values(ascending=False)
print(rev_by_region.to_string())


Q7: Region with highest total revenue
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09


In [9]:
# ── Q8 ───────────────────────────────────────────────────────────────────────
print("Q8: Payment method most used in cancelled orders")
cancelled = orders[orders['order_status'] == 'cancelled']
pm_cancelled = cancelled['payment_method'].value_counts()
print(pm_cancelled.to_string())

Q8: Payment method most used in cancelled orders
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535


In [10]:
# ── Q9 ───────────────────────────────────────────────────────────────────────
print("Q9: Size with highest return rate")
items_with_size   = order_items.merge(products[['product_id', 'size']], on='product_id')
returns_with_size = returns.merge(products[['product_id', 'size']], on='product_id')
items_count   = items_with_size.groupby('size').size()
returns_count = returns_with_size.groupby('size').size()
return_rate   = (returns_count / items_count).sort_values(ascending=False)
print(return_rate[['S', 'M', 'L', 'XL']].to_string())

Q9: Size with highest return rate
size
S     0.056515
M     0.055660
L     0.056250
XL    0.055200


In [11]:
# ── Q10 ──────────────────────────────────────────────────────────────────────
print("Q10: Installment plan with highest avg payment value")
avg_by_inst = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print(avg_by_inst.to_string())

Q10: Installment plan with highest avg payment value
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
